In [1]:
!pip install nltk

In [2]:
!pip install tensorflow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount= True)

In [ ]:
import pandas as pd

sentences = pd.read_csv("/content/drive/Shareddrives/NiiV/NiiV Projects/Multi-modal_interpretation/data/analysis/sentence_labeling/baseline-dataset/label_train.csv")
sentences['semantic_level'] = sentences['semantic_level'].apply(lambda x: [f"L{x}"])
sentences

# Splitting for train and test

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit


split = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)

for train_index, sample_index in split.split(sentences, sentences['semantic_level']):
    test_sentences = sentences.iloc[sample_index]


train_sentences = sentences.drop(test_sentences.index)


print(train_sentences.shape)
print(test_sentences.shape)

In [ ]:
from transformers import BertTokenizer


tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def encode_texts(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

# Tokenize Data
tokenized_data = encode_texts(train_sentences["sentence"].tolist())
input_ids = tokenized_data["input_ids"]
attention_mask = tokenized_data["attention_mask"]

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer
import torch

mlb = MultiLabelBinarizer()
labels = mlb.fit_transform(train_sentences["semantic_level"])
labels_tensor  = torch.tensor(labels, dtype=torch.float)

In [ ]:
import torch.nn as nn
from transformers import BertModel

class BiLSTMWithBERT(nn.Module):
    def __init__(self, hidden_dim, num_labels, unfreeze_last=2):
        super(BiLSTMWithBERT, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        for name, param in self.bert.named_parameters():
            param.requires_grad = False
            for layer_idx in range(12 - unfreeze_last, 12):
                if f"encoder.layer.{layer_idx}" in name or "pooler" in name:
                    param.requires_grad = True

        self.lstm    = nn.LSTM(
            input_size  = 768,
            hidden_size = hidden_dim,
            bidirectional= True,
            batch_first = True
        )
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = bert_out.last_hidden_state
        lstm_out, _ = self.lstm(x)
        pooled = torch.mean(lstm_out, dim=1)
        dropped = self.dropout(pooled)
        logits  = self.fc(dropped)
        return logits


In [ ]:
import numpy as np

class EarlyStopping:
    """Stops training when validation loss hasn’t improved for `patience` epochs."""
    def __init__(self, patience=3, min_delta=0.0, path="checkpoint.pth"):
        """
        Args:
            patience (int): How many epochs to wait after last time validation loss improved.
            min_delta (float): Minimum change to qualify as an improvement.
            path (str): Where to save the best model.
        """
        self.patience  = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter   = 0
        self.path      = path

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            # Improvement!
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.counter > self.patience:
                return True  # signal to stop
        return False

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import StepLR



dataset = TensorDataset(input_ids, attention_mask, labels_tensor)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

hidden_dim = 128
output_dim = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMWithBERT(hidden_dim, output_dim).to(device)


class_counts = train_sentences["semantic_level"].value_counts().sort_index().tolist()
class_weights = torch.tensor([sum(class_counts) / c for c in class_counts], dtype=torch.float)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)
early_stopper = EarlyStopping(patience=5, min_delta=1e-4,
                              path="best_model.pth")

for epoch in range(100):
    model.train()
    total_loss = 0
    for input_ids, attention_mask, label_batch in train_loader:
        input_ids, attention_mask, label_batch = (
            input_ids.to(device),
            attention_mask.to(device),
            label_batch.to(device)
        )
        optimizer.zero_grad()
        output = model(input_ids, attention_mask)
        loss = criterion(output, label_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}")

    if early_stopper(total_loss, model):
        print(f"Stopping early at epoch {epoch} (no improvement for {early_stopper.patience} epochs).")
        break


# Evaluation

In [ ]:
from sklearn.metrics import (
    hamming_loss,
    classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns


test_tokenized = encode_texts(test_sentences["sentence"].tolist())
test_input_ids = test_tokenized["input_ids"]
test_attention_mask = test_tokenized["attention_mask"]
label_tensors = torch.tensor(mlb.transform(test_sentences["semantic_level"]), dtype=torch.long)

dataset = TensorDataset(test_input_ids, test_attention_mask, label_tensors)
val_loader = DataLoader(dataset, batch_size=16, shuffle=True)

model.eval()
val_correct = 0
val_total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for input_ids, attention_mask, label_batch in val_loader:
        input_ids, attention_mask, label_batch = (
            input_ids.to(device),
            attention_mask.to(device),
            label_batch.to(device)
        )

    logits = model(test_input_ids.to(device), test_attention_mask.to(device))
    probs  = torch.sigmoid(logits).cpu().numpy()

y_true = label_tensors.numpy()
threshold = 0.5
y_pred = (probs >= threshold).astype(int)


ham_loss = hamming_loss(y_true, y_pred)
print(f"Hamming loss: {ham_loss:.4f}")
print(f"Hamming accuracy: {1 - ham_loss:.4f}")

target_names = [str(c) for c in mlb.classes_]
print("\nClassification report (per label):")
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=[str(c) for c in mlb.classes_],
    output_dict=True,
    zero_division=0
)

df_report = pd.DataFrame(report_dict).T

df_plot = df_report.drop(index=['micro avg', 'macro avg', 'weighted avg', 'samples avg'])

plt.figure(figsize=(8, 6))
sns.heatmap(
    df_plot.iloc[:, :3],
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={'label': 'Score'}
)
plt.title("Per-Label Precision / Recall / F1-Score")
plt.ylabel("Labels")
plt.xlabel("Metrics")
plt.tight_layout()
plt.show()

print(df_report.T[['micro avg', 'macro avg', 'weighted avg', 'samples avg']])

In [ ]:
checkpoint = {
    "hidden_dim":         hidden_dim,
    "num_labels":         output_dim,
    "num_categories":     num_categories,
    "unfreeze_last":      2,
    "mlb_classes":        mlb.classes_.tolist(),
    "tokenizer_name":     "bert-base-uncased",
    "model_state":        model.state_dict(),
    "optimizer_state":    optimizer.state_dict(),
    "scheduler_state":    scheduler.state_dict(),
    "epoch":              epoch,
}
torch.save(checkpoint, '/content/drive/Shareddrives/NiiV/NiiV Projects/Multi-modal_interpretation/data/analysis/sentence_labeling/automating_L1L4_label/bilstm_kwfreq.pth')
tokenizer.save_pretrained("/content/drive/Shareddrives/NiiV/NiiV Projects/Multi-modal_interpretation/data/analysis/sentence_labeling/automating_L1L4_label/bilstm_kwfreq/checkpoint/tokenizer/")

ckpt = torch.load("bilstm_bert_checkpoint.pth", map_location="cpu")

# 1) Rebuild the tokenizer
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("checkpoint/tokenizer/")

# 2) Recreate your MultiLabelBinarizer
from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
mlb.classes_ = ckpt["mlb_classes"]

# 3) Reconstruct the model
model = BiLSTMWithBERT(
    hidden_dim    = ckpt["hidden_dim"],
    num_labels    = ckpt["num_labels"],
    unfreeze_last = ckpt["unfreeze_last"]
)
model.load_state_dict(ckpt["model_state"])

# 4) (If you want to resume training) recreate optimizer & scheduler, then load their states
optimizer.load_state_dict(ckpt["optimizer_state"])
scheduler.load_state_dict(ckpt["scheduler_state"])

# 5) Move to device & set eval/train as needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
